# Neural Networks: Backpropagation with Multi-Layer Perceptrons

This project trains MLP classifiers and regressors (scikit-learn's `MLPClassifier`/`MLPRegressor`) across several datasets, covering overfit-avoidance techniques (early stopping, L2 regularization), hyperparameter tuning (learning rate, hidden layer size, momentum), automatic hyperparameter search (grid vs. randomized), and MLP regression with activation-function and architecture comparisons.

In [ ]:
from sklearn.neural_network import MLPClassifier, MLPRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

## Avoiding Overfit: Early Stopping and L2 Regularization

### Baseline (No Overfit Avoidance)

An `MLPClassifier` (one hidden layer of 64 nodes, logistic activation, SGD, no regularization) trained on the Iris dataset across 5 random 80/20 splits, reporting average iterations to convergence and train/test accuracy.

In [ ]:
#Iris with no regularization

# Load data set
train_data, _ = arff.loadarff("data/iris.arff")
iris_df = pd.DataFrame(train_data)

# Quick EDA
print("EDA:")
print("First 5 rows of Iris data set:")
display(iris_df.head())

print("\nIris Class Distributions: ")
display(iris_df["class"].value_counts().sort_index())

# X and y
X = iris_df.drop(["class"], axis = 1)
y = iris_df["class"]

# One-hot encode the categorical class column
y = pd.get_dummies(y)

# Set up the data table to be added to:
train_accuracies = []
test_accuracies = []
n_iters = []

# Do 5 random splits and train the model:
for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = 0.2
    )
    # train and fit the model
    mlp_classifier = MLPClassifier(
        hidden_layer_sizes = [64], 
        activation = "logistic",
        solver = "sgd",
        alpha = 0,
        batch_size = 1,
        learning_rate_init = 0.01,
        shuffle = True,
        momentum = 0,
        n_iter_no_change = 50,
        max_iter = 10000
    )
    mlp_classifier.fit(X_train, y_train)

    # predicitions:
    train_pred = mlp_classifier.predict(X_train)
    test_pred = mlp_classifier.predict(X_test)

    # get accuracies and num iters:
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    # num iters
    iters = mlp_classifier.n_iter_

    # add to lists
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    n_iters.append(iters)

    # for first run, print softmax probs
    if i == 0:
        print("\nSoft Max probs on test set:")
        print(mlp_classifier.predict_proba(X_test))

# turn into dictionaries, data frame, display
accuracies = {"Model": [i for i in range(1, 6)], 
              "Train Accuracy": train_accuracies, 
              "Test Accuracy": test_accuracies, 
              "Num. Iters": n_iters}
accuracies = pd.DataFrame(accuracies)
display(accuracies)

# Get averages:
avg_train = sum(train_accuracies) / len(train_accuracies)
avg_test = sum(test_accuracies) / len(test_accuracies)
avg_iter = sum(n_iters) / len(n_iters)

# average dictionary, data frame, display:
averages = {"Metric": ["Train Accuracy:", "Test Accuracy:", "Num. Iters:"], 
            "Average:" : [avg_train, avg_test, avg_iter]}
averages = pd.DataFrame(averages)
display(averages)

**Results**

Training took about a minute and a half across all 5 runs — faster than expected given the lab's setup (large hidden layer, high max-iteration ceiling). Across the 5 splits, average training accuracy was about 0.98 and average test accuracy about 0.96. The small train/test gap (2–3%) suggests only mild overfitting, if any — consistent with Iris being a simple, low-noise dataset that doesn't require aggressive regularization to generalize well.

### Early Stopping with a Validation Set

The same setup, with `early_stopping=True` and a 10% validation split used as the stopping criterion, plus a validation-accuracy-vs-epoch plot for one run.

In [ ]:
#Iris with early stopping and validation scores graph
# See problem above for EDA and splitting into X and y

train_accuracies = []
test_accuracies = []
n_iters = []
best_val_scores = []

# Do 5 random splits and train the model:
for i in range(5):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size = 0.2
    )
    # train and fit the model
    mlp_classifier = MLPClassifier(
        hidden_layer_sizes = [64], 
        activation = "logistic",
        solver = "sgd",
        alpha = 0,
        batch_size = 1,
        learning_rate_init = 0.01,
        shuffle = True,
        momentum = 0,
        n_iter_no_change = 50,
        max_iter = 10000,
        early_stopping = True,
        validation_fraction = 0.1, #internally split the training set into a validation set, too
    )
    mlp_classifier.fit(X_train, y_train)

    # predicitions:
    train_pred = mlp_classifier.predict(X_train)
    test_pred = mlp_classifier.predict(X_test)

    # get accuracies and num iters:
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    # num iters and best val scores
    iters = mlp_classifier.n_iter_
    best_val = mlp_classifier.best_validation_score_

    # add to lists / collect the metrics
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    n_iters.append(iters)
    best_val_scores.append(best_val)

    # for first run, plot the epochs vs accuracy on validation set
    if i == 0:
        # PLOT!!
        print("Validation Accuracy vs Epochs Graph for MLPClassifier 1:")
        plt.plot(range(1, len(mlp_classifier.validation_scores_)+1), mlp_classifier.validation_scores_)
        plt.xlabel("Epoch")
        plt.ylabel("Validation Accuracy")
        plt.title("Validation Accuracy vs Epochs")
        plt.show()

# turn into dictionaries, data frame, display
results = {"Model": [i for i in range(1, 6)], 
              "Train Accuracy": train_accuracies, 
              "Test Accuracy": test_accuracies, 
              "Num. Iters": n_iters,
              "Best Validation Score": best_val_scores}
results = pd.DataFrame(results)
print("\nMetrics:")
display(results)

# Get averages:
avg_train = sum(train_accuracies) / len(train_accuracies)
avg_test = sum(test_accuracies) / len(test_accuracies)
avg_iter = sum(n_iters) / len(n_iters)
avg_val_score = sum(best_val_scores) / len(best_val_scores)

# average dictionary, data frame, display:
averages = {"Metric": ["Train Accuracy:", "Test Accuracy:", "Num. Iters:", "Best Validation Score"], 
            "Average:" : [avg_train, avg_test, avg_iter, avg_val_score]}
averages = pd.DataFrame(averages)
print("\nAverages:")
display(averages)

**Results**

Both train and test accuracy dropped compared to the no-regularization baseline — averaging in the high-80s/low-90s rather than the high-90s. This isn't surprising given how small the Iris dataset is: early stopping carves out a validation set from an already-small training set, and less training data combined with high variance in that small validation set tends to hurt more than it helps here. Number of iterations to convergence was cut by more than half on average, as expected — training stops as soon as the validation score plateaus.

The validation-accuracy-vs-epoch plot was highly variable, with sharp spikes and dips (sometimes even flat stretches), a reasonable consequence of the very small dataset combined with a batch size of 1 — each epoch can shift the validation score substantially. Increasing batch size, or using a larger dataset, would likely smooth this out.

On a larger, noisier dataset, early stopping typically trades a small amount of training accuracy for improved test accuracy (better generalization) — the opposite of what's seen here, which is specific to how small and clean this dataset is.

### L2 Loss Regularization

The same base setup (no early stopping), sweeping L2 regularization strength (`alpha`) across [0.1, 0.01, 0.001, 0.0001, 0.00001], each averaged over 10 runs, followed by a training-loss-vs-epoch plot for the best-performing value.

In [ ]:
#Iris with Loss Regularization 
# See problem above for EDA and splitting into X and y

# Function for creating and fitting a mlp classifier
def mlp_func(alpha, X_train, y_train):
    """
    A function that initializes and fits an mlp classifier with a certain,
    predermined set of params, and a chosen alpha.

    Args:
        alpha (double): The alpha value to be used for the loss func
        X_train (np data frame): An data frame of all input features
        y_train (np series): A series of corresponding output features

    Returns:
        clf (MLPClassifier): A fitted MLP classifier
    """

    clf = MLPClassifier(
        hidden_layer_sizes = [64], 
        activation = "logistic",
        solver = "sgd",
        alpha = alpha, #get the alpha value
        batch_size = 1,
        learning_rate_init = 0.01,
        shuffle = True,
        momentum = 0,
        max_iter = 10000,
    )
    clf.fit(X_train, y_train)

    return clf

alphas = [0.1, 0.01, 0.001, 0.0001, .00001]

averages = []

# Do train the model 5 times for each of the alphas:
for alpha in alphas:
    train_accuracies = []
    test_accuracies = []
    n_iters = []
    best_losses = []
    for i in range(10):
        # train and fit the model
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size = 0.2, random_state = i)
        mlp_classifier = mlp_func(alpha, X_train, y_train)

        # predicitions:
        train_pred = mlp_classifier.predict(X_train)
        test_pred = mlp_classifier.predict(X_test)

        # get accuracies and num iters:
        train_acc = accuracy_score(y_train, train_pred)
        test_acc = accuracy_score(y_test, test_pred)

        # num iters and best val scores
        iters = mlp_classifier.n_iter_
        best_loss = mlp_classifier.best_loss_

        # add to lists / collect the metrics
        train_accuracies.append(train_acc)
        test_accuracies.append(test_acc)
        n_iters.append(iters)
        best_losses.append(best_loss)

    # Get averages:
    avg_train = sum(train_accuracies) / len(train_accuracies)
    avg_test = sum(test_accuracies) / len(test_accuracies)
    avg_iter = sum(n_iters) / len(n_iters)
    avg_loss = sum(best_losses) / len(best_losses)

    # average dictionary, data frame, display:
    averages.append({
        "Alpha": alpha,
        "Train Accuracy": avg_train,
        "Test Accuracy": avg_test,
        "Num Iterations": avg_iter,
        "Best Loss": avg_loss
    })

# Display averages 
averages = pd.DataFrame(averages)
print("\nAverages:")
display(averages)

# Get the best alpha value and then train the 
best_row = averages.loc[averages["Best Loss"].idxmin()]
best_alpha = best_row["Alpha"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 0)
mlp_classifier = mlp_func(best_alpha, X_train, y_train)

# # PLOT THE BEST ONE!!!!!!!!!
print("\nLoss vs Epochs graph:")
plt.plot(range(1, len(mlp_classifier.loss_curve_)+1), mlp_classifier.loss_curve_)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(f"Loss vs Epochs (alpha = {best_alpha})")
plt.show()

**Results**

The best-performing alpha, by loss, was 0.00001 — it tied 0.001 on test accuracy but achieved a lower loss, so it was selected as best on that basis. Interestingly, no regularization at all (`alpha=0` from the first experiment) achieved the single best test accuracy of the three approaches — though the gap to L2 regularization was small, and this may partly reflect a lucky single split in that first experiment versus the averaged results used here. Early stopping performed worst of the three, consistent with the earlier finding that it's poorly suited to a dataset this small.

**Comparing all three approaches:** on a dataset this small and clean, no regularization and light L2 regularization performed comparably well, while early stopping's validation-set requirement measurably hurt performance by further shrinking an already-small training set.

## Hyperparameter Tuning (Vowel Dataset)

### Baseline Accuracy and Feature Selection

Before tuning, it's worth establishing a naive baseline and identifying which input features shouldn't be used for training.

**Baseline accuracy:** Iris has 150 instances split evenly across 3 classes (50 each), so always predicting the majority class yields 50/150 ≈ 33.3% baseline accuracy. The Vowel dataset has 990 instances across 11 classes (90 each), giving a baseline of 90/990 ≈ 9.1%.

**Why Vowel's ceiling is lower:** with an even class distribution in both datasets, more output classes means each individual class makes up a smaller share of the data — so a model that's equally uncertain across classes has a much lower floor to beat on Vowel than on Iris. This is a direct consequence of the class-count difference, not a statement about how hard the underlying prediction task is.

**Features to exclude:** identifier-style fields like speaker number don't causally influence the vowel being spoken, so including them risks the model latching onto spurious patterns in an ID rather than genuine acoustic features — a classic case of a feature that can hurt generalization. Similarly, "Sex" and the provided train/test split indicator carry little direct predictive signal for this task and are dropped, since a custom train/test split is used instead of the dataset's own.

### Learning Rate

An MLP (hidden layer width = 2× input count) trained on the Vowel dataset across 5 learning rates spanning [0.001, 10], with a fixed 75/25 split and no early stopping.

In [ ]:
# Train with different learning rates
# Load arff

train_data, _ = arff.loadarff("data/vowel.arff")
vowel_df = pd.DataFrame(train_data)

# Quick EDA
print("Quick EDA of Vowels data set:")

print("\nFirst 5 rows of Vowel data set:")
display(vowel_df.head())

print("\nVowel Class Distributions: ")
display(vowel_df["Class"].value_counts().sort_index())

# X and y, drop columns said to be inapprporiate for training, decode/encode (?) classes
X = vowel_df.drop(["Train or Test", "Speaker Number", "Sex", "Class"], axis = 1)
vowel_df["Class"] = vowel_df["Class"].str.decode('utf-8')
le = LabelEncoder()
y = le.fit_transform(vowel_df["Class"])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.25,
    random_state = 3
)

lr = [0.001, 0.01, 0.1, 1, 10]
train_accs = []
test_accs = []
epochs = []

for i in range(len(lr)):
    # Initialize and fit Classifier
    mlp_clf = MLPClassifier(
        hidden_layer_sizes = [22],
        max_iter = 5000,
        learning_rate_init = lr[i]
    )
    mlp_clf.fit(X_train, y_train)

    # Predict
    train_pred = mlp_clf.predict(X_train)
    test_pred = mlp_clf.predict(X_test)

    # Score
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    # Epochs
    n_epochs = mlp_clf.n_iter_

    # Append
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    epochs.append(n_epochs)

# data frame, display
results = {
    "LR:": lr,
    "Num. Epochs": epochs,
    "Training Accuracy": train_accs,
    "Testing Accuracy:": test_accs
}
results = pd.DataFrame(results)
print("\nLearning rates of MLPClassifiers and their respective accuracies and n epochs:")
display(results)


**Results**

The largest learning rates tested (1 and 10) converged extremely fast — around 40 epochs — but at a steep cost: both train and test accuracy fell below 10%, by far the worst results seen across this whole project. Learning that fast effectively means taking huge, imprecise steps that overshoot good solutions rather than converging toward them.

Learning rates below 1 behaved far more reasonably, with training accuracy between 0.98–1.00 and test accuracy between 0.88–0.91. Epoch count grew roughly exponentially as learning rate decreased by factors of 10 (0.001 took about 2,700 epochs to converge, 0.01 took 862, and 0.1 took about 143) — smaller steps need proportionally more of them to reach the same destination.

**Learning rate 0.01** was selected as best: its train/test accuracy gap was small enough not to raise overfitting concerns, and it achieved the highest test accuracy among the reasonable candidates.

### Number of Hidden Nodes

Using the best learning rate found above, hidden-layer width was swept from 1 to 2048 (doubling each step) to find the point of diminishing returns.

In [ ]:
# Train with different numbers of hidden nodes
# See above part for a quick EDA of the data 

# Split the data - use a 80/20 split for this one.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.25,
    random_state = 3
)

hidden_nodes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
train_accs = []
test_accs = []
epochs = []

for i in range(len(hidden_nodes)):
    # Initialize and fit Classifier
    mlp_clf = MLPClassifier(
        hidden_layer_sizes = hidden_nodes[i],
        max_iter = 5000,
        learning_rate_init = 0.01,
        random_state = 3
    )
    mlp_clf.fit(X_train, y_train)

    # Predict
    train_pred = mlp_clf.predict(X_train)
    test_pred = mlp_clf.predict(X_test)

    # Score
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    # Epochs
    n_epochs = mlp_clf.n_iter_

    # Append
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    epochs.append(n_epochs)

# data frame, display
results = {
    "Hidden Nodes": hidden_nodes,
    "Num. Epochs": epochs,
    "Training Accuracy": train_accs,
    "Testing Accuracy:": test_accs
}
results = pd.DataFrame(results)
print("\n Number of Hidden nodes of MLPClassifiers and their respective accuracies and n epochs:")
display(results)

**Results**

Very small networks (1–4 hidden nodes) had low train and test accuracy and converged in relatively few epochs — too little capacity to capture the underlying patterns. Accuracy climbed steadily as node count increased, reaching 100% training accuracy by 32 nodes. Interestingly, mid-sized networks took *more* epochs (and thus more wall-clock time) to converge than either the smallest or largest networks tested — likely because they have enough capacity to start capturing complex patterns, but need more iterations to fully resolve them, whereas larger networks can converge on those same patterns more directly.

Test accuracy plateaued around 0.95–0.96, with little meaningful change between 128 and 1024 nodes, and a slight decline at 2048 (mild overfitting from unnecessary capacity). **128 hidden nodes** was carried forward as the practical choice — beyond that point, additional capacity bought essentially nothing.

### Momentum

Using the best hidden-layer size and learning rate found above, momentum was swept across [0.1, 0.3, 0.5, 0.7, 0.9] (note: this requires the SGD solver, which doesn't otherwise use momentum by default).

In [ ]:
# Train with different momentum values
# See above part for a quick EDA of the data 

# Split the data - use a 80/20 split for this one.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.25,
    random_state = 3
)

momentums = [0.1, 0.3, 0.5, 0.7, 0.9]
train_accs = []
test_accs = []
epochs = []

for i in range(len(momentums)):
    # Initialize and fit Classifier
    mlp_clf = MLPClassifier(
        hidden_layer_sizes = 128, # best number of hidden nodes
        max_iter = 5000,
        learning_rate_init = 0.01, # best lr
        solver = "sgd",
        momentum = momentums[i],
        random_state = 3
    )
    mlp_clf.fit(X_train, y_train)

    # Predict
    train_pred = mlp_clf.predict(X_train)
    test_pred = mlp_clf.predict(X_test)

    # Score
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    # Epochs
    n_epochs = mlp_clf.n_iter_

    # Append
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    epochs.append(n_epochs)

# data frame, display
results = {
    "Momentum": momentums,
    "Num. Epochs": epochs,
    "Training Accuracy": train_accs,
    "Testing Accuracy:": test_accs
}
results = pd.DataFrame(results)
print("\nMomentum values of MLPClassifiers and their respective accuracies and n epochs:")
display(results)

**Results**

As momentum increased from 0.1 to 0.9, test accuracy rose from about 0.87 to 0.94, and training accuracy rose from about 0.95 to 1.0. Notably, the number of epochs needed to converge *decreased* as momentum increased — higher momentum makes the optimizer less sensitive to small, noisy fluctuations in the gradient (it carries information from past steps forward), letting it move through flatter or noisier regions of the loss landscape more efficiently rather than getting stuck reacting to every local bump. That's consistent with the model achieving both its best accuracy *and* its fastest convergence at the highest momentum value tested.

### Automatic Hyperparameter Search: Grid vs. Randomized

Comparing `GridSearchCV` and `RandomizedSearchCV` for jointly tuning learning rate, hidden layer size, and momentum on the Vowel dataset.

In [ ]:
#Grid search for hyperparameters.
#Here is one variation of code you could use for your grid search. You can try your own variation if you prefer.
# Quick EDA
print("Quick EDA of Vowels data set:")

print("\nFirst 5 rows of Vowel data set:")
display(vowel_df.head())

print("\nVowel Class Distributions: ")
display(vowel_df["Class"].value_counts().sort_index())

from sklearn.model_selection import GridSearchCV
clf = MLPClassifier(activation='logistic', solver='sgd',alpha=0,early_stopping=True, n_iter_no_change=10, batch_size=1)
parameters = {'learning_rate_init':(0.001, 1), #You have to fill in the rest of your values for these lists
              'hidden_layer_sizes': ([8], [32], [128]),
              'momentum':(0.3, 0.5, 0.9)}
grid = GridSearchCV(clf, parameters)
grid.fit(X, y)    #This takes a while to run
print(grid.best_params_)
print(grid.best_score_)

In [ ]:
#Randomized search for hyperparameters
#Here is one variation of code you could use for your randomized search.

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
clf = MLPClassifier(activation='logistic', solver='sgd',alpha=0,early_stopping=True, n_iter_no_change=10, batch_size=1)
distributions = dict(learning_rate_init=uniform(loc=0.001, scale=10), #loc is the min val, and loc + scale is the max val
                    hidden_layer_sizes = ([8], [128]), #since there is no distribution it samples these values uniformly
                    momentum=uniform(loc=0,scale =.99))
search = RandomizedSearchCV(clf, distributions, n_iter=10)
search.fit(X, y)
print(search.best_params_)
print(search.best_score_)

**Results**

Grid search exhaustively tries every combination of the provided hyperparameter values, which makes it precise but expensive — its runtime grows roughly exponentially with the number of hyperparameters and values tested. Testing 3–4 values per hyperparameter across 3 hyperparameters here took anywhere from about 10 minutes to over 30, depending on the exact grid size.

Randomized search instead samples from a distribution of values a fixed number of times, trading exhaustiveness for speed — it finished in a little over 2 minutes here. It's especially well suited to hyperparameters that live on a continuous range (like learning rate) rather than a small discrete set, since it can explore that range without needing to pre-specify every value to test.

In practice, a reasonable workflow is randomized search first, to quickly narrow down a promising region of hyperparameter space, followed by a smaller, targeted grid search to fine-tune within that region.

One open question from this experiment: grid search and randomized search converged on noticeably different "best" hyperparameters here, despite searching similar spaces — worth investigating further, since the two methods might be finding different local optima in the accuracy landscape given the dataset's relative simplicity.

## MLP Regression

### Baseline Regression Model

An `MLPRegressor` (32 hidden nodes, SGD solver, momentum 0.9) trained on the red wine quality dataset, using standardized input features.

In [ ]:
# Load and Learn a real world regression data set
# To calculate MAE you could do a variation of the following

# Load the data
wine_df = pd.read_csv("data/winequality-red.csv", sep=';')

# Quick EDA
print("Quick EDA:")
print("\nFirst 5 rows of the Red Wine Quality dataset:")
display(wine_df.head())

print("\nWine Quality Distribution:")
display(wine_df["quality"].value_counts().sort_index())

# X and y
X = wine_df.drop(["quality"], axis = 1)
y = wine_df["quality"]

# Split into train and test using an 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 123
)

# Scale / standardize the X train values and X test values separately 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit a basic MLPRegressor (make an estimate of hyperparams to get a good **baseline** to compare to!)
mlp_reg_base = MLPRegressor(
    hidden_layer_sizes = [32],
    solver = "sgd",
    momentum = 0.9,
    n_iter_no_change = 25
)
mlp_reg_base.fit(X_train_scaled, y_train)


from sklearn.metrics import mean_absolute_error
print("Baseline mean_absolute error:")
print(mean_absolute_error(mlp_reg_base.predict(X_test_scaled), y_test))

**Approach**

The red wine quality dataset was chosen for its clean, fully numeric features and lack of missing values, keeping preprocessing simple and letting the focus stay on model tuning.

`MLPRegressor` differs from `MLPClassifier` in its output layer: linear activation and sum-of-squared-error loss (versus softmax activation and cross-entropy loss for classification) — configured for predicting a continuous value rather than a discrete class. A moderately large hidden layer (32 nodes) was chosen to balance accuracy against training time, and SGD with a fairly high momentum (0.9) was used based on the earlier finding that higher momentum improves both accuracy and convergence speed. `n_iter_no_change=25` (from the early-stopping experiments earlier) was also carried over, giving a modest MAE improvement.

**Results**

The baseline model achieved an MAE of about 0.52 — a reasonable starting point without extensive tuning.

### Activation Functions and Network Architecture

Comparing logistic, tanh, and ReLU activations across four hidden-layer configurations (single and two-layer, varying widths), using the Adam solver.

In [ ]:
# Run with different hyperparameters
# Experiment with hyperparams.

# See above problem for an EDA of this data set.

from sklearn.metrics import r2_score

activations = ["logistic", "tanh", "relu"]
layers = [
    (32,),
    (64,),
    (32, 16),
    (64, 32),
]

results = []

for act in activations:
    for layer in layers:
        model = MLPRegressor(
            activation=act,
            hidden_layer_sizes=layer,
            solver="adam",
            max_iter=1000,
            random_state=42
        )
        
        model.fit(X_train_scaled, y_train)

        # Predict
        preds = model.predict(X_test_scaled)

        # Metrics
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        
        # Append
        results.append({
            "Activation": act,
            "Layers": layer,
            "MAE": mae,
            "Test R2": r2
        })

# display results
results = pd.DataFrame(results)
print("MAE for various layer/node and acitvation function combinations:")
display(results)

**Results**

Against the 0.52 MAE baseline: **logistic** activation ranged from 0.49–0.53 MAE and 0.29–0.42 R², the widest R² range of the three — capable of the best individual result but the least consistent. **Tanh** was more stable (0.49–0.51 MAE, 0.32–0.41 R²), offering a better overall balance between accuracy and consistency. **ReLU** underperformed both (0.50–0.53 MAE, 0.28–0.35 R²), making it the weakest choice for this dataset among the three.

Architecture-wise, smaller/narrower networks consistently underperformed wider ones across every activation function tested. The best-performing configuration was the largest tested, (64, 32) — two hidden layers, 64 then 32 nodes — suggesting this dataset benefits from more model capacity rather than less.

The fairly wide variability in results across configurations suggests there's likely a better hyperparameter combination than what was tested here, and/or that the dataset itself doesn't offer enough signal for very tight, consistent generalization. A natural next step would be experimenting more systematically with solver choice, batch size, and learning-rate adaptation — all of which were left at defaults here (Adam was chosen for its general training stability, but not compared against alternatives in this pass).

## Conclusion

Across every experiment here, hyperparameter tuning showed clear diminishing returns past a certain point — more hidden nodes, more momentum, and lower learning rates all helped up to a threshold, beyond which they added training time without meaningfully improving accuracy. The overfit-avoidance techniques (early stopping, L2 regularization) were a useful reminder that "more regularization" isn't universally better — on a small, clean dataset like Iris, both actively hurt performance compared to no regularization at all, a result that would likely reverse on a larger, noisier dataset.

The grid-vs-randomized search comparison and the wine regression experiment both point toward the same next step: a more systematic, better-resourced hyperparameter search (larger grids, more randomized iterations, or a Bayesian optimization approach) would likely close the gap between the "reasonable" results found here and a genuinely well-tuned model.